### Task 5 - Model Iteration - BERT model


In [2]:
import pandas as pd
import re
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset

d:\anaconda\envs\block_c_y2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Load and clean the dataset
df = pd.read_csv("cleaned_balanced_dataset.csv", encoding="ISO-8859-1")
df = df.dropna(subset=["Corrected_Emotion"])


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df["Cleaned_Sentence"] = df["Corrected Sentence"].apply(clean_text)
df = df.drop_duplicates(subset=["Cleaned_Sentence", "Corrected_Emotion"])

In [15]:
# STEP 4: Label encode the target
le = LabelEncoder()
df["label"] = le.fit_transform(df["Corrected_Emotion"])

In [16]:
# STEP 5: Split into train and test sets
train_df, test_df = train_test_split(
    df[["Cleaned_Sentence", "label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=42,
)

In [17]:
# STEP 6: Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [18]:
# STEP 7: Tokenize text
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(example):
    return tokenizer(example["Cleaned_Sentence"], truncation=True)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 639/639 [00:00<00:00, 3620.28 examples/s]


In [19]:
# STEP 8: Calculate class weights for the loss function
from collections import Counter

label_counts = Counter(train_df["label"])
total = sum(label_counts.values())
weights = [total / label_counts[i] for i in range(len(label_counts))]
weights = torch.tensor(weights, dtype=torch.float)

In [20]:
# STEP 9: Load model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(le.classes_)
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
# ✅ STEP 10: Custom Trainer with fixed compute_loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [22]:
# STEP 11: Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_emotion_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

d:\anaconda\envs\block_c_y2\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
# STEP 12: Prepare data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [24]:
# STEP 13: Create trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

C:\Users\victo\AppData\Local\Temp\ipykernel_36548\2553590390.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


In [25]:
# STEP 14: Train
trainer.train()

# STEP 15: Evaluate
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("✅ Classification Report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))

                                       
  0%|          | 0/640 [09:49<?, ?it/s]          

{'loss': 1.9326, 'grad_norm': 9.051514625549316, 'learning_rate': 1.5000000000000002e-05, 'epoch': 1.0}










































                                       

                                         
  0%|          | 0/640 [09:59<?, ?it/s]          



{'eval_loss': 1.8769413232803345, 'eval_runtime': 10.3341, 'eval_samples_per_second': 61.834, 'eval_steps_per_second': 3.871, 'epoch': 1.0}


                                       
  0%|          | 0/640 [13:28<?, ?it/s]          

...

                                       
100%|██████████| 640/640 [14:34<00:00,  1.37s/it]


{'train_runtime': 874.4454, 'train_samples_per_second': 11.674, 'train_steps_per_second': 0.732, 'train_loss': 1.7522211074829102, 'epoch': 4.0}


100%|██████████| 40/40 [00:10<00:00,  3.99it/s]

✅ Classification Report:
              precision    recall  f1-score   support

       anger       1.00      0.06      0.12        16
     disgust       0.00      0.00      0.00         6
        fear       0.12      0.42      0.18        31
   happiness       0.25      0.41      0.31       110
     neutral       0.78      0.33      0.46       388
     sadness       0.15      0.37      0.21        35
    surprise       0.21      0.36      0.27        53

    accuracy                           0.34       639
   macro avg       0.36      0.28      0.22       639
weighted avg       0.57      0.34      0.38       639




d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
